# Text4GCN — Synthetic data demo

Offline-friendly guide: generate a tiny corpus, run the full pipeline, train a short GCN.

Run this notebook with the working directory set to `examples/` (or open it from that folder).


## 1. Setup


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "examples":
    PKG = ROOT.parent
    EXAMPLES = ROOT
else:
    # opened from repo root
    PKG = ROOT if (ROOT / "text4gcn").is_dir() else ROOT.parent
    EXAMPLES = PKG / "examples"

sys.path.insert(0, str(PKG))
sys.path.insert(0, str(EXAMPLES))

import text4gcn
from text4gcn import GNN, TextPipeline, Builder, Layer, GraphBuildStats
from text4gcn.builder import Frequency
from text4gcn.utils.device import check_pytorch_installation, get_device_manager
from synthetic_data_generator import SyntheticDataGenerator

print("text4gcn", text4gcn.__version__)
print(check_pytorch_installation())
print("device:", get_device_manager().device)
print("examples dir:", EXAMPLES)


## 2. Generate synthetic `.txt` / `.meta`


In [ ]:
DATA = EXAMPLES / "synthetic_data"
DATA.mkdir(exist_ok=True)

generator = SyntheticDataGenerator(str(DATA))
generator.generate_dataset("SyntheticR8", num_documents=400, train_ratio=0.8)

for f in sorted(DATA.glob("SyntheticR8.*")):
    print(f.name, f.stat().st_size, "bytes")


## 3. Preprocess


In [ ]:
TextPipeline(
    dataset_name="SyntheticR8",
    dataset_path=str(DATA),
    rare_count=3,
    language="english",
    seed=1234,
).execute(validation_ratio=0.10)

print("shuffled:", list((DATA / "SyntheticR8.shuffled").iterdir()))
print("features:", list((DATA / "SyntheticR8.node_features").iterdir()))


## 4. Build TF-IDF adjacency


In [ ]:
stats = Frequency(dataset_name="SyntheticR8", dataset_path=str(DATA)).tfidf(window_size=20)
print(stats)
assert isinstance(stats, GraphBuildStats)


## 5. Train (short run for demo)


In [ ]:
result = GNN(
    dataset_name="SyntheticR8",
    dataset_path=str(DATA),
    log_dir=str(EXAMPLES / "logs"),
    builder=Builder.Tfidf,
    layer=Layer.GCN,
    seed=1234,
    epochs=40,
    early_stopping=10,
    dropout=0.5,
    lr=0.02,
    nhid=200,
    device=None,
).fit()

print(result)
print(f"test accuracy = {result.test_accuracy:.4f}")


## 6. Convert your own documents

Text4GCN expects:

- `{name}.txt` — one document per line
- `{name}.meta` — `index\tsplit\tlabel` (`train` / `test`)


In [ ]:
def convert_documents_to_text4gcn_format(documents, labels, train_ratio=0.8, output_dir="custom_data", dataset_name="custom_dataset", seed=1234):
    import random
    from pathlib import Path

    if len(documents) != len(labels):
        raise ValueError("documents and labels must have the same length")

    rng = random.Random(seed)
    pairs = list(zip(documents, labels))
    rng.shuffle(pairs)
    documents, labels = zip(*pairs)

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    n_train = int(len(documents) * train_ratio)

    with open(out / f"{dataset_name}.txt", "w", encoding="utf-8") as f:
        for doc in documents:
            f.write(doc.strip().replace("\n", " ") + "\n")

    with open(out / f"{dataset_name}.meta", "w", encoding="utf-8") as f:
        for i, label in enumerate(labels):
            split = "train" if i < n_train else "test"
            f.write(f"{i}\t{split}\t{label}\n")

    return str(out)

sample_documents = [
    "machine learning artificial intelligence algorithms neural networks",
    "company quarterly earnings revenue growth market share",
    "research study medical treatment clinical trial outcomes",
    "sports team championship performance athletic training",
    "technology innovation digital transformation software cloud",
    "scientific methodology analysis hypothesis experiment results",
    "business strategy customer satisfaction sales expansion",
    "athletic training endurance strength competition league",
    "healthcare professionals diagnosis therapy patient recovery",
    "computer programming coding software development systems",
]
sample_labels = [
    "technology", "business", "science", "sports", "technology",
    "science", "business", "sports", "health", "technology",
]

custom_dir = convert_documents_to_text4gcn_format(
    sample_documents,
    sample_labels,
    output_dir=str(EXAMPLES / "custom_data"),
    dataset_name="custom_dataset",
)
print("wrote", custom_dir)

try:
    TextPipeline(
        dataset_name="custom_dataset",
        dataset_path=custom_dir,
        rare_count=1,
        language="english",
        seed=1234,
    ).execute(validation_ratio=0.10)
    print("custom TextPipeline OK")
except Exception as exc:
    print("custom pipeline note:", exc)


## Summary

You exercised the same three steps as production use:

1. prepare `.txt` / `.meta`
2. `TextPipeline` + `Frequency.tfidf`
3. `GNN(...).fit()` → `TrainingResult`

For the R8 benchmark walkthrough see `notebooks/text4gcn.ipynb`.
